In [1]:
#ADAPTING PRE-TRAINED LANGUAGE MODELS FOR QUANTUM NATURAL LANGUAGE PROCESSING
# https://arxiv.org/abs/2302.13812

In [7]:
# -*- coding: utf-8 -*-
import torch
import pickle
import os
from discopy.tensor import Dim
from datasets import load_dataset
from lambeq import AtomicType, SpiderAnsatz, PytorchTrainer, Dataset,PytorchModel, BobcatParser
print('running discocat classification on sst...')
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-3
# Otherwise there will be weird bugs in converting the diagrams to tensor networks
model_dim = 2
nb_classes = 2
SEED = 0
max_train_samples = 10000
max_val_samples = 10000
dataset = load_dataset('glue','sst2')

running discocat classification on sst...


In [ ]:
dataset['train']['sentence']

In [ ]:
dataset['train']['sentence']

In [5]:
max_train_samples = min(max_train_samples, len(dataset['train']['sentence']))
max_val_samples = min(max_val_samples, len(dataset['validation']['sentence']))
parser = BobcatParser(verbose='text')

train_diagrams = parser.sentences2diagrams(dataset['train']['sentence'][:max_train_samples])
val_diagrams = parser.sentences2diagrams(dataset['validation']['sentence'][:max_val_samples])
train_labels= [[float(y), float(1-y)] for y in dataset['train']['label'][:max_train_samples]]
val_labels = [[float(y), float(1-y)] for y in dataset['validation']['label'][:max_val_samples]]

max_train_samples = min(max_train_samples, len(train_diagrams))
max_val_samples = min(max_val_samples, len(val_diagrams))

Tagging sentences.
Parsing tagged sentences.


BobcatParseError: Bobcat failed to parse 'irritates and'.

In [ ]:
new_train_diagrams = []
new_train_labels = []
for i, (diagram, label) in enumerate(zip(train_diagrams, train_labels)):
    if i >=max_train_samples:
        break
    # Some sentences are not grammatically correct,
    # and the parser will return None result
    if diagram is not None:
        new_train_diagrams.append(diagram)
        new_train_labels.append(label)
del train_diagrams
del train_labels

In [ ]:
new_val_diagrams = []
new_val_labels = []
for i, (diagram, label) in enumerate(zip(val_diagrams, val_labels)):
    if i >=max_val_samples:
        break
    if diagram is not None:
        new_val_diagrams.append(diagram)
        new_val_labels.append(label)
del val_diagrams
del val_labels

In [ ]:
# Convert the diagrams to circuits
ansatz = SpiderAnsatz({AtomicType.NOUN: Dim(model_dim),
    AtomicType.SENTENCE: Dim(nb_classes),
    AtomicType.PREPOSITIONAL_PHRASE: Dim(model_dim)})

In [ ]:
train_circuits = []
val_circuits = []
legal_ids = []
for i, diagram in enumerate(new_train_diagrams):
    # Some diagrams cannot be converted to circuits,
    # and None will be returned
    try:
        circuit = ansatz(diagram)
        train_circuits.append(circuit)
        legal_ids.append(i)
    except:
        continue

In [ ]:
train_labels = [new_train_labels[x] for x in legal_ids]
legal_ids = []
for i, diagram in enumerate(new_val_diagrams):
    try:
        circuit = ansatz(diagram)
        val_circuits.append(circuit)
        legal_ids.append(i)
    except:
        continue

val_labels = [new_val_labels[x] for x in legal_ids]
all_circuits = train_circuits + val_circuits

In [ ]:
model = PytorchModel.from_diagrams(all_circuits)
def accuracy(y_hat, y):
    return sum(torch.argmax(y_hat,dim=-1) == torch.argmax(y, dim=-1))/len(y)

eval_metrics = {"acc": accuracy}

# Create Trainer and datasets from the pre-processed circuits
trainer = PytorchTrainer(
    model=model,
    loss_function=torch.nn.CrossEntropyLoss(),
    optimizer=torch.optim.AdamW,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    evaluate_functions=eval_metrics,
    evaluate_on_train=True,
    verbose='text',
    seed=SEED)

print('creating dataset...')

train_dataset = Dataset(    
    train_circuits,
    train_labels,
    batch_size=BATCH_SIZE)

val_dataset = Dataset(val_circuits, val_labels, shuffle=False)

In [ ]:
# Training the model
print('training...')
trainer.fit(train_dataset, val_dataset, evaluation_step=1, logging_step=1
)

In [ ]:
import matplotlib.pyplot as plt
fig1, ((ax_tl, ax_tr), (ax_bl, ax_br)) = plt.subplots(2, 2, sharey='row',figsize=(10, 6))

ax_tl.set_title('Training set')
ax_tr.set_title('Development set')
ax_bl.set_xlabel('Epochs')
ax_br.set_xlabel('Epochs')
ax_bl.set_ylabel('Accuracy')
ax_tl.set_ylabel('Loss')

colours = iter(plt.rcParams['axes.prop_cycle'].by_key()['color'])
ax_tl.plot(trainer.train_epoch_costs, color=next(colours))
ax_bl.plot(trainer.train_results['acc'], color=next(colours))
ax_tr.plot(trainer.val_costs, color=next(colours))
ax_br.plot(trainer.val_results['acc'], color=next(colours))

# print validation accuracy
test_output = model(val_circuits)
test_labels = torch.tensor(val_labels)

if type(test_output) == tuple:
    test_output, legal_idx = test_output
    test_labels = test_labels[legal_idx]
    test_acc = accuracy(test_output, test_labels)

with open('acc_discocat_sst2.txt','w') as writer:
    writer.write(str(test_acc.item()))

Tagging sentences.


: 